#### This is a sample notebook on Implementing and evaluating the results of DPO on LLMs and the difference it can make to LLM reasoning

#### Steps to achieve it
* Create sample datasets, Each dataset contains full conversation history of a LLM Agent (why agent ? because this will involve tools) 
* Inspect the Dataset and decide if the LLM answer is preffered and if it is not then make the preffered answer and if it is then make the non preffered answer
* Finally perform DPO with the generated data and Evaluate

In [1]:
import pandas as pd
import json
from utils.initialize_os_agents import OSAgentsInitializer
from utils.keys import set_api_keys
from utils.tools import human_send_message
set_api_keys()
import geopandas as gpd
import shutil
import joblib
import os
from pathlib import Path
import numpy as np
from openai import OpenAI
from pydantic import BaseModel, Field, Json
from typing import List, Any

Openai and ngd key set successfully
Openai and ngd key set successfully


### Here we are making the llm prompt to simulate some of the trajectories. We need preffered and non preffered responses so 
* We will check if given 1 trajectory is a powerful model able to generate the good and not obvious unpreffered response
* We will also try to see if we can make some progess in automatically generating preffered and non preffered trajectories

In [2]:
llm_generate_trajectories = """ You are given data trajectories generated by a system specializing in geospatial operations \
  specifically these tasks are generated by the orchestrator module.
  
  <Trajectory Features>
  1. Each trajectory has a question in the begining and you will see how the host approaches the problem by delegating it to other agents and tools \
  2. Thus the trajectory shows the functioning of the host agent and how it communicates using subagents \
  3. You will see `send_message` tool used to call other agents and other tools used to inspect the data \
  4. You will also see the responses and final outputs \
  5. You will see a multi turn conversation occuring between tools \
  6. The data that is helping the geospatial tool is OpenStreet map data for Devon. \
  <Trajectory Features> \
  
  <Underlying data> \
  1. The underlying data is openstreet map data for devon with the following databases \
        a. devon_boundaries : used by named_area subagent \
        b. devon_buildings : used by buildings subagent \
        c. devon_landuse : used by landuse subagent \
        d. devon_natural :used by land agent \
        e. devon_pois : used by address agent \
        f. devon_waterways : used by water_network agent \
  2. All of these above subagents controll the databases and are orchestrated by host whose trajectory is presented. \
  <underlying data> \
  
  <Your Tasks>
  1. Look at the input Trajectory given below \
  2. Look at the databases called using the `send_message_tool` \
  3. Generate a different query (both in location and task) that is centered around the recognised databases in the input trajectory \
  4. Generate a new trajectory for the query \
  5. Repeat above 4 times to get 4 trajectories and output is a list of 4 json trajectories \
  <Your Tasks>

### COMPLEXITY INSERTIONS:
1. In 1 out of 4 trajectories induce some complexity related to locations and tasks:
  a.  Multiple locations founds -> ask human -> generate human response -> query location database with human response -> generate new response \
  b. Query multiple relevant databases for the same task like schools in buildings or land_use etc

  ### Database Enforcement Logic
1. ANALYZE: Scan the <Input Trajectory> for any 'send_message' calls. 
2. EXTRACT: Identify which specific Devon databases (e.g., devon_buildings, devon_waterways) are being accessed.
3. RESTRICT: Your 4 new queries MUST use the exact same database(s) found in Step 2. 
4. FORBIDDEN: Do NOT use any other databases from the 'Underlying data' list if they were not in the input. 
   - Example: If input only uses 'devon_natural', all 4 new trajectories must only use 'devon_natural' for different tasks/locations.

  
  Input Trajectory : <Input Trajectory>

  """

class Trajectories(BaseModel):
    trajectories : list[Json[Any]] =  Field(description = "A list of 4 json trajectories with similar format as input")

In [3]:
llm_extract_trajectories = """You are given a raw log from a multi-agent system.

Your task is to extract a **clean trajectory of the HOST agent only** and convert it into a DPO-compatible format.

---

## IMPORTANT

* Focus ONLY on the HOST agent.
* Ignore internal workings of other agents.
* Use `send_message` calls as the primary interaction mechanism.
* The `target` field indicates which agent the host is communicating with.

---

## INPUT LOG

<LOG_FILE>

---

## TASK

1. Identify all actions performed by the HOST agent.
2. Extract:

   * user query
   * messages sent via `send_message`
   * tool calls made by the host
   * responses received that influence the host
3. Reconstruct a chronological sequence of steps.
4. Ignore debug logs, internal messages of other agents, and irrelevant noise.

---

## OUTPUT FORMAT (STRICT JSON)

Return Example:

{
"messages": [
 {"role": "user", "content": "Turn on the living room lights."},
    {"role": "assistant", "tool_calls": [
        {"type": "function", "function": {
            "name": "control_light",
            "arguments": {"room": "living room", "state": "on"}
        }}]
    },
    {"role": "tool", "name": "control_light", "content": "The lights in the living room are now on."},
    {"role": "assistant", "content": "Done!"}

]
}

---

## RULES

* Do NOT include other agents as separate roles
* Do NOT include internal reasoning unless explicitly present in host logs
* Merge multiple log events into coherent steps
* Preserve order of execution
* Keep the assistant response aligned with the trajectory
* Output must be valid JSON only

---

## NOTES

* Treat each `send_message(target=...)` as a host action
* Responses from other agents should be treated as observations
* The final output should represent the host’s complete solution

"""

In [4]:
def generate_synthetic_trajectories(system_prompt, json_log):

    client = OpenAI()
    json_log_path = Path.cwd()/ "DPO_dumps"/"DPO_trajectories"/ json_log

    json_log_content = None
    
    with open(json_log_path,"r") as file:
        json_log_content = json.load(file)
    
    system_prompt = system_prompt.replace("<Input Trajectory>",str(json_log_content))

    messages = [{"role":"user","content":system_prompt}]

    response = client.beta.chat.completions.parse(model = "gpt-4.1", temperature=0, messages=messages, response_format=Trajectories)

    trajectories = response.choices[0].message.parsed

    return trajectories


In [9]:
synthetic_trajectories = {}
for file in os.listdir(r"DPO_dumps\DPO_trajectories"):
    
    print(f"Attempting synthetic data generation for {file}")
    try:
        trajectories = generate_synthetic_trajectories(llm_generate_trajectories, file)
    except Exception as e:
        print(f"Error synthetic data generation for {file} with error {e}")
        continue
    synthetic_trajectories[file] = trajectories
    

Attempting synthetic data generation for trajectory_1.json
Attempting synthetic data generation for trajectory_10.json
Attempting synthetic data generation for trajectory_11.json
Attempting synthetic data generation for trajectory_12.json
Attempting synthetic data generation for trajectory_13.json
Error synthetic data generation for trajectory_13.json with error 1 validation error for Trajectories
trajectories.0
  Invalid JSON: expected `,` or `}` at line 1 column 1458 [type=json_invalid, input_value='{"messages": [{"role": "... need more details!"}]}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Attempting synthetic data generation for trajectory_14.json
Error synthetic data generation for trajectory_14.json with error 4 validation errors for Trajectories
trajectories.0
  Invalid JSON: EOF while parsing an object at line 1 column 2733 [type=json_invalid, input_value='{"messages": [{"role": "... please let me know!"}]', input_type=s

In [13]:
for file_name,output in synthetic_trajectories.items():
    for trajectory_num,trajectory in enumerate(output.trajectories):
        with open(f"DPO_dumps/DPO_synthetic/synthetic_{trajectory_num}_{file_name}","w") as file:
            json.dump(trajectory,file,indent=2)

In [18]:
def extract_trajectories(system_prompt, log_file_name):

    client = OpenAI()
    log_file_path = Path.cwd() / "evaluation" / "evaluation_logs" / log_file_name
    log_file_content = None
    
    with open(log_file_path,"rb") as file:
        log_file_content = file.read()

    log_file_content = log_file_content.decode("cp1252")
        
    system_prompt = system_prompt.replace("<LOG_FILE>",log_file_content)
    messages = [{"role":"user","content":system_prompt}]
    #response = client.chat.completions.create(model = "o3-mini", messages = messages).choices[0].message.content
    response = client.chat.completions.create(model = client.models.list().data[0].id, messages = messages, temperature=0).choices[0].message.content
    return response



def extract_trajectories_opensource(system_prompt:str, log_file_name:str):

    '''Helps to generate trajectories from modes
    Args:
        system_prompt: Generates trajectories using system prompt
        log_file_name: Gets more info from log file name
    '''

    client = OpenAI(base_url = "http://localhost:8001/v1",api_key="dummy")
    log_file_path = Path.cwd() / "evaluation" / "evaluation_logs" / log_file_name
    log_file_content = None
    
    with open(log_file_path,"rb") as file:
        log_file_content = file.read()

    log_file_content = log_file_content.decode("cp1252")
        
    system_prompt = system_prompt.replace("<LOG_FILE>",log_file_content)
    messages = [{"role":"user","content":system_prompt}]
    #response = client.chat.completions.create(model = "o3-mini", messages = messages).choices[0].message.content
    response = client.chat.completions.create(model = client.models.list().data[0].id, messages = messages, temperature=0).choices[0].message.content
    return response




In [5]:
trajectories = []
for question in os.listdir(r"evaluation\evaluation_logs"):
    try:
        print(f"Attempting to analyze log {question}")
        trajectory = extract_trajectories_opensource(llm_extract_trajectories,question)
        cleaned_trajectory = trajectory.strip().strip('`').replace('json\n', '', 1)
        cleaned_trajectory = json.loads(cleaned_trajectory)
        trajectories.append(cleaned_trajectory)
    except Exception as e:
        print(f"Failed in analysing log {question} due to {e}")


Attempting to analyze log Question_0.log
Attempting to analyze log Question_1.log
Attempting to analyze log Question_10.log
Attempting to analyze log Question_11.log
Attempting to analyze log Question_12.log
Attempting to analyze log Question_13.log
Attempting to analyze log Question_14.log
Attempting to analyze log Question_15.log
Failed in analysing log Question_15.log due to Error code: 400 - {'error': {'message': "You passed 65001 input tokens and requested 0 output tokens. However, the model's context length is only 65000 tokens, resulting in a maximum input length of 65000 tokens. Please reduce the length of the input prompt. (parameter=input_tokens, value=65001)", 'type': 'BadRequestError', 'param': 'input_tokens', 'code': 400}}
Attempting to analyze log Question_16.log
Failed in analysing log Question_16.log due to Error code: 400 - {'error': {'message': "You passed 65001 input tokens and requested 0 output tokens. However, the model's context length is only 65000 tokens, resul

In [8]:
for num,trajectory in enumerate(trajectories):
    with open(f"DPO_dumps/DPO_trajectories/trajectory_{num}.json","w") as file:
        json.dump(trajectory,file)

In [40]:
for entry in d:
    print(entry)

{'messages': [{'role': 'user', 'content': 'What information do you have on wetlands in sowton?'}, {'role': 'assistant', 'tool_calls': [{'type': 'function', 'function': {'name': 'send_message', 'arguments': {'target': 'planning_agent', 'task_description': 'The user wants information on wetlands in Sowton (UK). Outline the high-level sequence of steps and which agents to use to obtain and plot this information, following the framework rules.'}}}], 'content': ''}, {'role': 'tool', 'name': 'send_message', 'content': 'output steps: ["Find Sowton 1 area", "Search for wetlands in Sowton as many search results", "Plot the wetlands on a map"]\n\nAGENTS TO USE:\n- Geographic Information System (GIS) tools to search for and plot wetlands.\n- Local environmental databases or government resources that provide information on wetlands in the UK.\n- Mapping software to visualize the wetlands in Sowton.'}, {'role': 'assistant', 'tool_calls': [{'type': 'function', 'function': {'name': 'send_message', 'a

In [41]:
var = """{"role": "assistant", "content": "{\n  \"reasoning\": \"Constructing logic chain: CurrentYearThreshold for startup, ConstantScenario for dry surcharge, AnnualGrowth for efficiency, and StorageThreshold + IndexedArray for drought cuts.\",\n  \"added_parameters\": {\n    \"zenith_base_flow\": {\"type\": \"constant\", \"value\": 12.0},\n    \"zenith_efficiency_growth\": {\"type\": \"AnnualGrowthParameter\", \"start_year\": 2028, \"rate\": -0.02},\n    \"zenith_scenario_surcharge\": {\"type\": \"constantscenario\", \"scenario\": \"hydrological_ensembles\", \"values\": [0.0, 3.5, 0.0]},\n    \"zenith_gross_demand\": {\"type\": \"aggregated\", \"agg_func\": \"sum\", \"parameters\": [\"zenith_base_flow\", \"zenith_scenario_surcharge\"]},\n    \"zenith_commissioning_factor\": {\"type\": \"aggregated\", \"agg_func\": \"sum\", \"parameters\": [\"trigger_2027\", \"trigger_2029\"]},\n    \"zenith_drought_threshold\": {\"type\": \"storagethreshold\", \"storage_node\": \"Opal_Reservoir\", \"threshold\": 45.0, \"predicate\": \"LT\"},\n    \"zenith_restriction_factor\": {\"type\": \"indexedarray\", \"index_parameter\": \"zenith_drought_threshold\", \"parameters\": [\"zenith_thermal_normal_multiplier\", \"zenith_thermal_drought_multiplier\"]},\n    \"zenith_final_max_flow\": {\"type\": \"aggregated\", \"agg_func\": \"product\", \"parameters\": [\"zenith_gross_demand\", \"zenith_efficiency_growth\", \"zenith_commissioning_factor\", \"zenith_restriction_factor\"]}\n  },\n  \"custom_parameter\": \"from pywr.parameters import Parameter\\n\\nclass AnnualGrowthParameter(Parameter):\\n    def __init__(self, model, rate, start_year, base_value=1.0, **kwargs):\\n        super().__init__(model, **kwargs)\\n        self.rate = float(rate)\\n        self.start_year = int(start_year)\\n        self.base_value = float(base_value)\\n    def value(self, timestep, scenario_index):\\n        current_year = timestep.datetime.year\\n        if current_year < self.start_year:\\n            return self.base_value\\n        years_elapsed = current_year - self.start_year\\n        return self.base_value * (1.0 + self.rate) ** years_elapsed\\n\\nAnnualGrowthParameter.register()\"\n}"}"""
print(var)

{"role": "assistant", "content": "{
  "reasoning": "Constructing logic chain: CurrentYearThreshold for startup, ConstantScenario for dry surcharge, AnnualGrowth for efficiency, and StorageThreshold + IndexedArray for drought cuts.",
  "added_parameters": {
    "zenith_base_flow": {"type": "constant", "value": 12.0},
    "zenith_efficiency_growth": {"type": "AnnualGrowthParameter", "start_year": 2028, "rate": -0.02},
    "zenith_scenario_surcharge": {"type": "constantscenario", "scenario": "hydrological_ensembles", "values": [0.0, 3.5, 0.0]},
    "zenith_gross_demand": {"type": "aggregated", "agg_func": "sum", "parameters": ["zenith_base_flow", "zenith_scenario_surcharge"]},
    "zenith_commissioning_factor": {"type": "aggregated", "agg_func": "sum", "parameters": ["trigger_2027", "trigger_2029"]},
    "zenith_drought_threshold": {"type": "storagethreshold", "storage_node": "Opal_Reservoir", "threshold": 45.0, "predicate": "LT"},
    "zenith_restriction_factor": {"type": "indexedarray",